#Creating dim table for customers

In [0]:
CREATE OR REPLACE TABLE 03_prod_gold.analytics.dim_customers AS
SELECT DISTINCT
    customerkey,
    gender,
    -- name,
    city,
    state_code,
    state,
    -- zip_code,
    country,
    continent,
    birthday
FROM 02_prod_silver.transform.customers;

# Creating dim table products

In [0]:
CREATE OR REPLACE TABLE 03_prod_gold.analytics.dim_products AS
SELECT DISTINCT
    productkey,
    product_name,
    brand,
    color,
    subcategory,
    category,
    unit_cost_usd,
    unit_price_usd
FROM 02_prod_silver.transform.products;

# Creating dim table for stores

In [0]:
CREATE OR REPLACE TABLE 03_prod_gold.analytics.dim_stores AS
SELECT DISTINCT
    storekey,
    country,
    state,
    square_meters,
    open_date
FROM 02_prod_silver.transform.stores;

# Creating dim table for date

In [0]:
CREATE OR REPLACE TABLE 03_prod_gold.analytics.dim_date AS
SELECT DISTINCT
    order_date AS date,
    YEAR(order_date) AS year,
    MONTH(order_date) AS month,
    DAY(order_date) AS day
FROM 02_prod_silver.transform.sales;

# Creating fact table 


In [0]:
CREATE OR REPLACE TABLE 03_prod_gold.analytics.fact_sales AS
SELECT
    s.order_number,
    s.line_item,
    s.order_date,
    s.delivery_date,
    s.customerkey,
    s.storekey,
    s.productkey,
    s.quantity,
    s.currency_code,

    p.unit_price_usd,

    -- Revenue
    (s.quantity * p.unit_price_usd / COALESCE(e.exchange_rate, 1)) AS revenue_usd,

    -- Cost
    (s.quantity * p.unit_cost_usd / COALESCE(e.exchange_rate, 1)) AS cost_usd,

    -- Profit
    ((s.quantity * p.unit_price_usd - s.quantity * p.unit_cost_usd) 
        / COALESCE(e.exchange_rate, 1)) AS profit_usd,

    -- Delivery time
    DATEDIFF(COALESCE(s.delivery_date, s.order_date), s.order_date) AS delivery_days

FROM 02_prod_silver.transform.sales s
LEFT JOIN 02_prod_silver.transform.products p
    ON s.productkey = p.productkey
LEFT JOIN 02_prod_silver.transform.exchange_rate e
    ON s.order_date = e.date
    AND s.currency_code = e.currency_code;

In [0]:

SELECT *
FROM 03_prod_gold.analytics.fact_sales;

#KPI

#1. Monthly Revenue Trend
## Calculate total revenue USD by month 

In [0]:
SELECT 
    YEAR(order_date) AS year,
    MONTH(order_date) AS month,
    SUM(revenue_usd) AS total_revenue_usd
FROM 03_prod_gold.analytics.fact_sales
WHERE YEAR(order_date) = 2020
GROUP BY YEAR(order_date), MONTH(order_date)
ORDER BY month;

#2. Peak Month Analysis
## identify top 3 revenue months. Calculate % of annual total.

In [0]:
WITH monthly_rev AS (
    SELECT 
        YEAR(order_date) AS year,
        MONTH(order_date) AS month,
        SUM(revenue_usd) AS revenue_usd
    FROM 03_prod_gold.analytics.fact_sales
    WHERE YEAR(order_date) = 2020
    GROUP BY YEAR(order_date), MONTH(order_date)
),

total_rev AS (
    SELECT SUM(revenue_usd) AS total_revenue
    FROM monthly_rev
)

SELECT 
    m.month,
    m.revenue_usd,
    ROUND((m.revenue_usd / t.total_revenue) * 100, 2) AS pct_of_total
FROM monthly_rev m
CROSS JOIN total_rev t
ORDER BY m.revenue_usd DESC
LIMIT 3;

# 3. Holiday Drivers
##peak months, show top 3 product categories driving revenue.

In [0]:
WITH monthly_rev AS (
    SELECT 
        MONTH(order_date) AS month,
        SUM(revenue_usd) AS revenue_usd
    FROM 03_prod_gold.analytics.fact_sales
    WHERE YEAR(order_date) = 2020
    GROUP BY MONTH(order_date)
),

top_months AS (
    SELECT month
    FROM monthly_rev
    ORDER BY revenue_usd DESC
    LIMIT 3
),

category_rev AS (
    SELECT 
        p.category,
        SUM(f.revenue_usd) AS category_revenue
    FROM 03_prod_gold.analytics.fact_sales f
    JOIN 03_prod_gold.analytics.dim_products p
        ON f.productkey = p.productkey
    WHERE YEAR(f.order_date) = 2020
      AND MONTH(f.order_date) IN (SELECT month FROM top_months)
    GROUP BY p.category
),

total_peak AS (
    SELECT SUM(category_revenue) AS total_revenue
    FROM category_rev
)

SELECT 
    c.category,
    c.category_revenue,
    ROUND((c.category_revenue / t.total_revenue) * 100, 2) AS pct_of_peak_total
FROM category_rev c
CROSS JOIN total_peak t
ORDER BY c.category_revenue DESC
LIMIT 3;

#4. Delivery Performance
##Calculate overall avg delivery time across all orders.



In [0]:
SELECT 
    ROUND(AVG(delivery_days), 2) AS average_days,
    COUNT(DISTINCT order_number) AS total_orders
FROM 03_prod_gold.analytics.fact_sales
WHERE delivery_days IS NOT NULL
  AND delivery_days > 0;

#5. Country Delivery Issues
## Avg delivery time by store country. Show slowest 5 countries.


In [0]:
SELECT 
    COALESCE(s.country, 'ONLINE') AS country,
    
    ROUND(AVG(f.delivery_days), 2) AS avg_days,
    
    COUNT(DISTINCT f.order_number) AS order_count,
    
    percentile_approx(f.delivery_days, 0.5) AS median_days

FROM 03_prod_gold.analytics.fact_sales f

LEFT JOIN 03_prod_gold.analytics.dim_stores s
    ON f.storekey = s.storekey   -- LEFT JOIN to keep online orders
-- WHERE f.delivery_days > 0
GROUP BY COALESCE(s.country, 'ONLINE')

ORDER BY avg_days DESC
LIMIT 5;

#6. Channel Performance
## AOV (revenue/orders) by Online vs In-Store across continents (Online = StoreKey.isna()).


In [0]:
SELECT
    c.continent,

    --  Online (storekey = 0 OR NULL)
    ROUND(
        SUM(CASE WHEN f.storekey = 0 OR f.storekey IS NULL THEN f.revenue_usd END) /
        COUNT(DISTINCT CASE WHEN f.storekey = 0 OR f.storekey IS NULL THEN f.order_number END),
    2) AS AOV_online,

    --  In-Store
    ROUND(
        SUM(CASE WHEN f.storekey != 0 AND f.storekey IS NOT NULL THEN f.revenue_usd END) /
        COUNT(DISTINCT CASE WHEN f.storekey != 0 AND f.storekey IS NOT NULL THEN f.order_number END),
    2) AS AOV_store,

    -- Order counts
    COUNT(DISTINCT CASE WHEN f.storekey = 0 OR f.storekey IS NULL THEN f.order_number END) AS online_orders,

    COUNT(DISTINCT CASE WHEN f.storekey != 0 AND f.storekey IS NOT NULL THEN f.order_number END) AS store_orders

FROM 03_prod_gold.analytics.fact_sales f

JOIN 03_prod_gold.analytics.dim_customers c
    ON f.customerkey = c.customerkey

GROUP BY c.continent
ORDER BY c.continent;

#7. Volume Leaders
##Top 5 product categories by total units sold.

In [0]:
WITH category_units AS (
    SELECT 
        p.category,
        SUM(f.quantity) AS units_sold
    FROM 03_prod_gold.analytics.fact_sales f
    JOIN 03_prod_gold.analytics.dim_products p
        ON f.productkey = p.productkey
    GROUP BY p.category
),

total_units AS (
    SELECT SUM(units_sold) AS total_units
    FROM category_units
)

SELECT 
    RANK() OVER (ORDER BY c.units_sold DESC) AS rank,
    c.category,
    c.units_sold,
    ROUND((c.units_sold / t.total_units) * 100, 2) AS percent_of_total_units

FROM category_units c
CROSS JOIN total_units t

ORDER BY rank
LIMIT 5;

#8. Revenue Leaders
##Top 5 product categories by total revenue USD.


In [0]:
WITH category_revenue AS (
    SELECT 
        p.category,
        SUM(f.revenue_usd) AS revenue_usd
    FROM 03_prod_gold.analytics.fact_sales f
    JOIN 03_prod_gold.analytics.dim_products p
        ON f.productkey = p.productkey
    GROUP BY p.category
),

total_revenue AS (
    SELECT SUM(revenue_usd) AS total_rev
    FROM category_revenue
)

SELECT 
    RANK() OVER (ORDER BY c.revenue_usd DESC) AS rank,
    c.category,
    ROUND(c.revenue_usd, 2) AS revenue_usd,
    ROUND((c.revenue_usd / t.total_rev) * 100, 2) AS percent_of_total_revenue

FROM category_revenue c
CROSS JOIN total_revenue t

ORDER BY rank
LIMIT 5;

# 9. Customer Profile
## Customer count and spending by Continent × Gender.

In [0]:
SELECT 
    c.continent,
    c.gender,

    COUNT(DISTINCT c.customerkey) AS customer_count,

    ROUND(SUM(f.revenue_usd), 2) AS total_spend_usd,

    ROUND(
        SUM(f.revenue_usd) / COUNT(DISTINCT c.customerkey), 
    2) AS avg_spend_per_cust

FROM 03_prod_gold.analytics.fact_sales f

JOIN 03_prod_gold.analytics.dim_customers c
    ON f.customerkey = c.customerkey

GROUP BY c.continent, c.gender
ORDER BY c.continent, c.gender;

#10. Customer Loyalty
## Repeat customer rate (% with 2+ orders) by continent.


In [0]:
WITH customer_orders AS (
    SELECT 
        customerkey,
        COUNT(DISTINCT order_number) AS order_count
    FROM 03_prod_gold.analytics.fact_sales
    GROUP BY customerkey
),

repeat_customers AS (
    SELECT customerkey
    FROM customer_orders
    WHERE order_count >= 2
)

SELECT 
    c.continent,

    -- Total unique customers
    COUNT(DISTINCT c.customerkey) AS unique_customers,

    -- Repeat customers
    COUNT(DISTINCT r.customerkey) AS repeat_customers,

    -- Repeat rate %
    ROUND(
        COUNT(DISTINCT r.customerkey) * 100.0 
        / COUNT(DISTINCT c.customerkey),
    2) AS repeat_rate_percent

FROM 03_prod_gold.analytics.dim_customers c

LEFT JOIN repeat_customers r
    ON c.customerkey = r.customerkey

WHERE c.customerkey != -1

GROUP BY c.continent
ORDER BY c.continent;